In [8]:
import json
from pathlib import Path

import pandas as pd

# ContraDoc

> Li, J., Raheja, D., & Kumar, D. (2024). *ContraDoc: Understanding Self-Contradictions in Documents with Large Language Models*. NAACL 2024. https://arxiv.org/abs/2311.09182

CONTRADOC contains 449 self-contradictory (positive) and 442 non-contradictory (negative) documents sourced from CNN/DailyMail News, Wikipedia, and Story Summaries; document length varies from 300 to 2200 tokens. Positive examples are synthesized by inserting or replacing a single sentence in an otherwise consistent document; every modification is human-verified.

## Positive example schema

| field | meaning |
|---|---|
| `text` | full document |
| `evidence` | the sentence that introduces the contradiction |
| `doc_type` | `story`, `news`, or `wiki` |
| `contra_plug` | `Insert` or `Replace` |
| `contra_type` | list of contradiction types |
| `scope` | `global`, `local`, or `intra` |
| `ref sentences` (optional) | sentences the evidence contradicts |
| `unique id` | per-document identifier |

## Negative example schema

Only `text`, `doc_type`, `unique id`.

## This notebook

Loads `data/raw/ContraDoc.json`, flattens `pos` + `neg` into a single dataframe with stripped `text` and `evidence`, and saves as `data/processed/ContraDoc.csv` (449 YES + 442 NO = 891 rows). `contra_type`, `scope`, `contra_plug`, and `ref_sentences` are pipe-joined when list-valued; NO rows get `"none"` placeholders for the YES-only fields.

## Load raw JSON

In [9]:
raw_path = Path("data/raw/ContraDoc/ContraDoc.json")
with raw_path.open("r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"pos: {len(raw['pos'])} documents")
print(f"neg: {len(raw['neg'])} documents")

pos: 449 documents
neg: 442 documents


## Flatten `pos` + `neg` into one dataframe

Output columns: `id`, `contradiction` (YES/NO), `doc_type`, `scope`, `contra_plug`, `contra_type`, `evidence`, `ref_sentences`, `text`.

- `text` and `evidence` are `.strip()`-ed to match the RealContra convention.
- List-valued fields (`contra_type`, and occasionally `scope` / `contra_plug`) are pipe-joined (`|`).
- NO rows get `"none"` for YES-only categorical fields and `""` for `evidence`.

In [10]:
def _listify(value) -> str:
    if value is None:
        return "none"
    if isinstance(value, list):
        return "|".join(str(v) for v in value) if value else "none"
    return str(value)


def _strip(text):
    return text.strip() if isinstance(text, str) else text


pos_rows = []
for doc_id, entry in raw["pos"].items():
    pos_rows.append(
        {
            "id": entry.get("unique id") or doc_id,
            "contradiction": "YES",
            "doc_type": _listify(entry.get("doc_type")),
            "scope": _listify(entry.get("scope")),
            "contra_plug": _listify(entry.get("contra_plug")),
            "contra_type": _listify(entry.get("contra_type")),
            "evidence": _strip(entry.get("evidence", "")),
            "ref_sentences": _listify(entry.get("ref sentences")),
            "text": _strip(entry.get("text", "")),
        }
    )

neg_rows = []
for doc_id, entry in raw["neg"].items():
    neg_rows.append(
        {
            "id": entry.get("unique id") or doc_id,
            "contradiction": "NO",
            "doc_type": _listify(entry.get("doc_type")),
            "scope": "none",
            "contra_plug": "none",
            "contra_type": "none",
            "evidence": "",
            "ref_sentences": "none",
            "text": _strip(entry.get("text", "")),
        }
    )

contradoc_df = pd.concat([pd.DataFrame(pos_rows), pd.DataFrame(neg_rows)], ignore_index=True)
print(f"Total rows: {len(contradoc_df)}")
print(f"Columns: {list(contradoc_df.columns)}")
contradoc_df.head()

Total rows: 891
Columns: ['id', 'contradiction', 'doc_type', 'scope', 'contra_plug', 'contra_type', 'evidence', 'ref_sentences', 'text']


,id,contradiction,doc_type,scope,contra_plug,contra_type,evidence,ref_sentences,text
0,3488771835_6,YES,story,local,Replace,Perspective/View/Opinion|Emotion/Mood/Feeling,"One rat, Jenner, agreed wholeheartedly with Th...",none,Mrs. Frisby is the widowed head of a family of...
1,3488771840_2,YES,story,intra,Replace,Causal,"It is a simple story, and one unlikely to appe...",none,Mrs. Tittlemouse is a tale in which no humans ...
2,3488771840_5,YES,story,local,Replace,Perspective/View/Opinion|Causal,Mrs. Tittlemouse tries to pull out their nest ...,none,Mrs. Tittlemouse is a tale in which no humans ...
3,3488771840_7,YES,story,intra,Replace,Emotion/Mood/Feeling,Mrs. Tittlemouse feels calm and collected as a...,none,Mrs. Tittlemouse is a tale in which no humans ...
4,3488771840_9,YES,story,intra,Replace,Causal,The fastidious little mouse spends only a day ...,none,Mrs. Tittlemouse is a tale in which no humans ...


## Distributions

In [11]:
print("Contradiction label:")
print(contradoc_df["contradiction"].value_counts())
print("\nDoc type (all):")
print(contradoc_df["doc_type"].value_counts())
print("\nScope (YES only):")
print(contradoc_df[contradoc_df["contradiction"] == "YES"]["scope"].value_counts())
print("\nContradiction plug (YES only):")
print(contradoc_df[contradoc_df["contradiction"] == "YES"]["contra_plug"].value_counts())
print("\nContradiction types (YES only, expanded):")
yes_types = contradoc_df[contradoc_df["contradiction"] == "YES"]["contra_type"].str.split("|").explode()
print(yes_types.value_counts())

Contradiction label:
contradiction
YES    449
NO     442
Name: count, dtype: int64

Doc type (all):
doc_type
news     312
wiki     295
story    284
Name: count, dtype: int64

Scope (YES only):
scope
local            220
global           155
intra             73
global, intra      1
Name: count, dtype: int64

Contradiction plug (YES only):
contra_plug
Replace    271
Insert     178
Name: count, dtype: int64

Contradiction types (YES only, expanded):
contra_type
Content                     288
Perspective/View/Opinion    101
Negation                     87
Emotion/Mood/Feeling         86
Numeric                      65
Factual                      54
Causal                       36
Relation                     25
Other                         1
Name: count, dtype: int64


## Save `ContraDoc.csv`

In [12]:
output_path = Path("data/processed/ContraDoc/ContraDoc.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
contradoc_df.to_csv(output_path, index=False)
print(f"Saved {len(contradoc_df)} rows -> {output_path.resolve()}")

Saved 891 rows -> D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\ContraDoc\ContraDoc.csv
